# Chapter 7 — Tool and Interface Engineering with MCP
## AI-Based Data Engineering (Packt)

This notebook shows the SQL patterns that power the three MCP servers in Chapter 7:

- **`catalog_server`** — read-only schema context (tools: `get_table_schema`, `search_catalog`, `get_column_lineage`)
- **`analytics_server`** — read-only aggregate queries against OpsPulse marts
- **`operations_server`** — lineage and access-history queries

**Note:** MCP servers are long-running processes that expose tools over the MCP wire protocol. They cannot run inside a notebook cell. This notebook shows the underlying Snowflake queries each tool executes, so you can verify, profile, and tune them independently of the server. The final cells demonstrate the ACI principle and provide local setup instructions.

## The ACI Principle

**ACI** (Agent-Computer Interface by design): every sentence in a tool's name, description, and parameter descriptions is a behavioral instruction to the model.

- **Ambiguous documentation** → ambiguous model behavior
- **Prescriptive documentation** → deterministic tool selection

The `catalog_server` server-level instruction illustrates this:

```
"Always call get_table_schema before generating SQL or proposing schema changes.
 The catalog is the authoritative source of column definitions."
```

This single instruction prevents the model from generating SQL from training knowledge — it always fetches current schema first. The SQL cells below are exactly what each tool executes on Snowflake.

In [ ]:
%%sql -r table_schema_result
-- catalog_server: get_table_schema('OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS')
-- The MCP tool parses this result into a structured JSON response.
-- ACI note: the tool description says 'ALWAYS call this before generating SQL'
-- so the model never infers column names from training knowledge.
SELECT
    column_name,
    data_type,
    is_nullable,
    COALESCE(comment, '[no description]') AS description
FROM OPSPU.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'MARTS'
  AND table_name   = 'FCT_ACTIVE_CUSTOMERS'
ORDER BY ordinal_position;

In [ ]:
%%sql -r catalog_search_result
-- catalog_server: search_catalog('active customers') — LIKE fallback
-- In production the tool calls a Cortex Search index for hybrid BM25+semantic ranking.
-- This LIKE query is the fallback for environments without a Cortex Search service.
SELECT
    table_name,
    table_type,
    row_count,
    COALESCE(comment, '') AS comment
FROM OPSPU.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'MARTS'
  AND (table_name ILIKE '%customer%' OR table_name ILIKE '%active%')
ORDER BY row_count DESC NULLS LAST;

In [ ]:
%%sql -r analytics_result
-- analytics_server: safe read-only aggregate query
-- The analytics_server enforces SELECT-only by validating the query before execution.
-- NOTE: requires OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS from the OpsPulse generator.
SELECT
    region_code,
    COUNT(*)              AS active_customers,
    AVG(total_orders_30d) AS avg_orders_30d
FROM OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS
GROUP BY region_code
ORDER BY active_customers DESC;

In [ ]:
%%sql -r lineage_result
-- catalog_server: get_query_access_history for FCT_ACTIVE_CUSTOMERS
-- (Full column lineage requires ACCOUNT_USAGE.ACCESS_HISTORY)
-- Queries SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY (30-day window).
-- Requires ACCOUNTADMIN or explicit SNOWFLAKE.ACCOUNT_USAGE read grant.
-- Returns an empty result set if access is denied — no error.
SELECT DISTINCT
    user_name,
    query_type,
    DATE(query_start_time) AS query_date
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE query_text ILIKE '%FCT_ACTIVE_CUSTOMERS%'
  AND query_start_time > DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY query_date DESC
LIMIT 20;

In [ ]:
from snowflake.snowpark.context import get_active_session
import anthropic

session = get_active_session()
client  = anthropic.Anthropic()

# The ACI principle: every description IS a behavioral instruction.
# 'ALWAYS call this before generating SQL' forces schema fetch before SQL generation.
tools = [
    {
        'name': 'get_table_schema',
        'description': (
            'Return column schema for a table. '
            'ALWAYS call this before generating SQL or proposing schema changes. '
            'Never infer column names or types from training knowledge — '
            'the catalog is the authoritative source.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'table_fqn': {
                    'type': 'string',
                    'description': (
                        'Fully-qualified table name: DB.SCHEMA.TABLE. '
                        'Example: OPSPU.MARTS.FCT_ACTIVE_CUSTOMERS'
                    ),
                },
            },
            'required': ['table_fqn'],
        },
    },
]

response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=300,
    tools=tools,
    messages=[{
        'role': 'user',
        'content': 'How many active customers are in the EMEA region?',
    }],
)

if response.stop_reason == 'tool_use':
    tool_use = next(b for b in response.content if b.type == 'tool_use')
    print(f'Model called tool: {tool_use.name}')
    print(f'Tool input:        {tool_use.input}')
    print('\nThe ACI principle is working: '
          'the model fetched schema before generating SQL.')
else:
    print(f'Stop reason: {response.stop_reason}')
    print(response.content[0].text[:300])

## Running the MCP Servers Locally

Each server is a standalone Python script that exposes tools over the MCP wire protocol.

**Prerequisites:**
```bash
pip install mcp anthropic snowflake-connector-python
export SNOWFLAKE_ACCOUNT=<your-account>
export SNOWFLAKE_USER=<your-user>
export SNOWFLAKE_PASSWORD=<your-password>
```

**Start the catalog server:**
```bash
cd code/ch07_mcp_tools
python catalog_server.py
```

**Claude Desktop configuration** (`~/Library/Application Support/Claude/claude_desktop_config.json`):
```json
{
  "mcpServers": {
    "opspu-catalog": {
      "command": "python",
      "args": ["/path/to/code/ch07_mcp_tools/catalog_server.py"]
    }
  }
}
```

Once connected, Claude will automatically call `get_table_schema` before any SQL generation — the same behavior shown in the ACI demo cell above.

> **External Access Integration required.** This notebook calls the Anthropic API from Snowflake. Your admin must:
> 1. Create an External Access Integration for `api.anthropic.com`
> 2. Set `ANTHROPIC_API_KEY` as a Snowflake Secret and attach it to this notebook
>
> Without this, the `anthropic.Anthropic()` calls will fail. See [Snowflake EAI docs](https://docs.snowflake.com/en/developer-guide/external-network-access/creating-using-external-network-access).


## Summary

The three MCP servers enforce four behavioral layers:

| Layer | Server | What it prevents |
|---|---|---|
| **Schema grounding** | `catalog_server` | SQL hallucinated from training knowledge |
| **Read-only analytics** | `analytics_server` | Accidental DML from a broad-permission agent |
| **Lineage check** | `catalog_server` | Schema changes without impact assessment |
| **Platform capabilities** | `catalog_server` | Agents calling endpoints that don't exist |

The server-level `instructions=` string in `FastMCP` is the highest-priority behavioral constraint — it applies before any tool call and sets the model's operating context for the entire session. See `code/ch07_mcp_tools/` for the full three-server implementation.